1. Cluster the documents into 2 groups


In [1]:
# Install dependencies
!pip install -q sentence-transformers scikit-learn

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Documents
documents = [
    "The plaintiff filed a lawsuit for breach of contract.",
    "Contract disputes are common in civil litigation.",
    "The defendant denies all allegations of negligence.",
    "Personal injury and negligence claims require proof of harm."
]

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Convert text to embeddings
embeddings = model.encode(documents)

# Apply KMeans clustering
kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(embeddings)

# Print results
for i, doc in enumerate(documents):
    print(f"Document {i+1}: Cluster {labels[i]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Document 1: Cluster 0
Document 2: Cluster 0
Document 3: Cluster 1
Document 4: Cluster 1


2. Measure similarity (doc1 vs doc2, doc1 vs doc3)

In [2]:
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Documents
doc1 = "The tenant failed to pay rent on time, violating the lease agreement."
doc2 = "The lease was breached because the monthly rent payment was late."
doc3 = "The company merged with a competitor to dominate the market."

# Encode
emb1 = model.encode(doc1, convert_to_tensor=True)
emb2 = model.encode(doc2, convert_to_tensor=True)
emb3 = model.encode(doc3, convert_to_tensor=True)

# Compute cosine similarity
sim_12 = util.cos_sim(emb1, emb2)
sim_13 = util.cos_sim(emb1, emb3)

print(f"Similarity (doc1 vs doc2): {sim_12.item():.4f}")
print(f"Similarity (doc1 vs doc3): {sim_13.item():.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similarity (doc1 vs doc2): 0.7678
Similarity (doc1 vs doc3): 0.0145


3. LLM-style semantic similarity (SentenceTransformer-based)

In [3]:
from sentence_transformers import SentenceTransformer, util

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Documents
doc1 = "The tenant failed to pay rent on time, violating the lease agreement."
doc2 = "The lease was breached because the monthly rent payment was late and not sent to the appropriate email."
doc3 = "The company merged with a competitor to dominate the market."

# Encode all at once
embeddings = model.encode([doc1, doc2, doc3], convert_to_tensor=True)

# Compute similarities
sim_12 = util.cos_sim(embeddings[0], embeddings[1])
sim_13 = util.cos_sim(embeddings[0], embeddings[2])

print("---- Similarity Scores ----")
print(f"doc1 vs doc2: {sim_12.item():.4f}")
print(f"doc1 vs doc3: {sim_13.item():.4f}")

# Optional interpretation
def interpret(score):
    if score > 0.7:
        return "Highly similar"
    elif score > 0.4:
        return "Moderately similar"
    else:
        return "Not similar"

print("\n---- Interpretation ----")
print(f"doc1 vs doc2: {interpret(sim_12.item())}")
print(f"doc1 vs doc3: {interpret(sim_13.item())}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


---- Similarity Scores ----
doc1 vs doc2: 0.6553
doc1 vs doc3: 0.0145

---- Interpretation ----
doc1 vs doc2: Moderately similar
doc1 vs doc3: Not similar
